# ❄️ SNOW Data Product Pipeline
## Domain: IT Service Management | Catalog: `snow_product` | Owner: ITSM Team

This notebook runs the complete **Bronze → Silver → Gold** pipeline for the SNOW domain's data product: **Service Health**.

**Pipeline Flow:**
```
ServiceNow CSVs → Bronze (raw_incidents, raw_change_requests)
                     ↓
                  Silver (incidents, change_requests) — enriched + validated
                     ↓
                  Gold (service_health) — aggregated per application
```

**Data Product Characteristics demonstrated:**
- ✅ **Clear Owner** — ITSM Team owns the catalog and all transformations
- ✅ **Business Purpose** — Service health metrics for SLA reviews
- ✅ **Documented Definitions** — Every column has a COMMENT
- ✅ **Quality Built-In** — Inline validation at silver layer

## 🪨 Bronze Layer — Raw Ingestion from ServiceNow
Raw ticket data extracted via ServiceNow Table API. Preserves original field values.

In [ ]:
# Load CSV data into bronze tables with explicit schema casting
import os, pandas as pd

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
bundle_root = notebook_path.rsplit('/src/notebooks/', 1)[0]
data_dir = f"/Workspace{bundle_root}/src/data"

if not os.path.exists(data_dir):
    data_dir = "/Workspace/Users/skarotirajashekar@godevsuite060.onmicrosoft.com/dbx-dp-snow/src/data"

print(f"Data directory: {data_dir}")

for csv_file, table_name in [
    ("snow_incidents.csv", "snow_product.bronze.raw_incidents"),
    ("snow_change_requests.csv", "snow_product.bronze.raw_change_requests")
]:
    csv_path = f"{data_dir}/{csv_file}"
    pdf = pd.read_csv(csv_path)
    df = spark.createDataFrame(pdf)
    # Cast to match table schema (CSV infers int64/Long, table expects INT)
    from pyspark.sql.types import IntegerType
    for col_name, col_type in df.dtypes:
        if col_type == 'bigint':
            df = df.withColumn(col_name, df[col_name].cast(IntegerType()))
    df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
    print(f"Loaded {len(pdf)} rows -> {table_name}")


In [ ]:
%sql
-- Verify bronze: Raw incidents from ServiceNow
SELECT incident_id, short_description, priority, state, category, 
       assigned_to, affected_application_id, created_at, resolved_at, sla_breached
FROM snow_product.bronze.raw_incidents
ORDER BY created_at DESC

In [ ]:
%sql
-- Verify bronze: Raw change requests from ServiceNow
SELECT change_id, description, change_type, risk, state,
       affected_application_id, requested_by, success
FROM snow_product.bronze.raw_change_requests
ORDER BY planned_start DESC

## ⚙️ Silver Layer — Curated & Enriched
Transformations applied:
1. **Derived Fields** — `priority_label` (human-readable), `resolution_hours`, `is_open`
2. **Standardization** — `TRIM()`, `INITCAP()` on text fields
3. **Calculated Metrics** — `duration_hours`, `on_schedule`
4. **Validation** — Priority 1–4, non-null incident IDs

In [ ]:
%sql
-- Silver transformation for incidents
MERGE INTO snow_product.silver.incidents AS target
USING (
    SELECT 
        incident_id, TRIM(short_description) AS short_description, priority,
        CASE priority WHEN 1 THEN 'Critical' WHEN 2 THEN 'High' WHEN 3 THEN 'Medium' WHEN 4 THEN 'Low' END AS priority_label,
        state, INITCAP(category) AS category, assigned_to, assignment_group, affected_application_id,
        created_at, resolved_at,
        ROUND(TIMESTAMPDIFF(MINUTE, created_at, COALESCE(resolved_at, current_timestamp())) / 60.0, 2) AS resolution_hours,
        sla_breached,
        CASE WHEN state IN ('New', 'In Progress', 'On Hold') THEN true ELSE false END AS is_open,
        current_timestamp() AS processed_at
    FROM snow_product.bronze.raw_incidents
    WHERE incident_id IS NOT NULL AND priority BETWEEN 1 AND 4
) AS source
ON target.incident_id = source.incident_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [ ]:
%sql
-- Silver transformation for change requests
MERGE INTO snow_product.silver.change_requests AS target
USING (
    SELECT 
        change_id, TRIM(description) AS description, change_type, risk,
        state, affected_application_id, requested_by,
        planned_start, planned_end, success,
        ROUND(TIMESTAMPDIFF(MINUTE, planned_start, planned_end) / 60.0, 2) AS duration_hours,
        current_timestamp() AS processed_at
    FROM snow_product.bronze.raw_change_requests
    WHERE change_id IS NOT NULL
) AS source
ON target.change_id = source.change_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [ ]:
%sql
-- Verify silver: Enriched incidents
SELECT incident_id, short_description, priority_label, state, category,
       affected_application_id, resolution_hours, sla_breached, is_open
FROM snow_product.silver.incidents
ORDER BY priority, created_at DESC

## 🏆 Gold Layer — Service Health Data Product
Aggregates incidents and changes **per application** to produce the **Service Health** data product. This is what IT Operations and management consume for SLA reviews.

In [ ]:
%sql
-- Gold: Build Service Health data product by aggregating per application
MERGE INTO snow_product.gold.service_health AS target
USING (
    WITH incident_metrics AS (
        SELECT 
            affected_application_id,
            COUNT(*) AS total_incidents,
            SUM(CASE WHEN is_open THEN 1 ELSE 0 END) AS open_incidents,
            SUM(CASE WHEN priority = 1 THEN 1 ELSE 0 END) AS critical_incidents,
            SUM(CASE WHEN priority = 2 THEN 1 ELSE 0 END) AS high_incidents,
            SUM(CASE WHEN sla_breached THEN 1 ELSE 0 END) AS sla_breaches,
            ROUND(SUM(CASE WHEN NOT sla_breached THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS sla_compliance_pct,
            ROUND(AVG(resolution_hours), 2) AS avg_resolution_hours
        FROM snow_product.silver.incidents
        GROUP BY affected_application_id
    ),
    change_metrics AS (
        SELECT 
            affected_application_id,
            COUNT(*) AS total_changes,
            ROUND(SUM(CASE WHEN success = true THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(CASE WHEN success IS NOT NULL THEN 1 END), 0), 1) AS change_success_rate_pct
        FROM snow_product.silver.change_requests
        GROUP BY affected_application_id
    )
    SELECT 
        i.affected_application_id,
        i.total_incidents, i.open_incidents, i.critical_incidents, i.high_incidents,
        i.sla_breaches, i.sla_compliance_pct, i.avg_resolution_hours,
        COALESCE(c.total_changes, 0) AS total_changes,
        COALESCE(c.change_success_rate_pct, 100.0) AS change_success_rate_pct,
        -- Risk score: higher = riskier
        (i.critical_incidents * 10 + i.high_incidents * 5 + i.sla_breaches * 8 
         + CASE WHEN COALESCE(c.change_success_rate_pct, 100) < 90 THEN 10 ELSE 0 END) AS risk_score,
        current_timestamp() AS product_generated_at
    FROM incident_metrics i
    LEFT JOIN change_metrics c ON i.affected_application_id = c.affected_application_id
) AS source
ON target.affected_application_id = source.affected_application_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [ ]:
%sql
-- The final data product: Service Health per application
SELECT affected_application_id, total_incidents, open_incidents, critical_incidents,
       sla_breaches, sla_compliance_pct, ROUND(avg_resolution_hours, 1) AS avg_hours,
       total_changes, change_success_rate_pct, risk_score
FROM snow_product.gold.service_health
ORDER BY risk_score DESC, total_incidents DESC

## ✅ Pipeline Complete
**Service Health** data product is populated:
- **Bronze**: 12 incidents + 8 change requests (raw from ServiceNow)
- **Silver**: 12 incidents + 8 change requests (enriched, validated)
- **Gold**: 10 application health records (aggregated, scored)

Next: Run `02_Snow_Governance` for quality checks, contracts, CDF, UDFs, and sharing.